In [1]:
import pandas as pd

train = pd.read_csv("../data/raw/train.csv")
test = pd.read_csv("../data/raw/test.csv")

print(train.shape, test.shape)
train.head()

(15000, 14) (10000, 13)


,id,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,0,15648876.0,Shih,652.0,France,Male,31.0,9.0,129901.21,1.0,0.0,0.0,120531.31,0.0
1,1,15662494.0,Okeke,812.0,France,Male,38.0,5.0,0.00,2.0,1.0,1.0,185547.74,0.0
2,2,15655238.0,Trevisani,710.0,Germany,Female,23.0,1.0,121376.15,1.0,1.0,1.0,57589.29,0.0
3,3,15752300.0,Yashina,558.0,Spain,Female,36.0,10.0,0.00,2.0,1.0,1.0,126192.54,0.0
4,4,15770548.0,Ndubuagha,625.0,Germany,Male,39.0,4.0,137592.41,1.0,1.0,0.0,76946.60,1.0


In [3]:
train.isnull().sum()

id                 0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

In [4]:
test.isnull().sum()

id                 0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
dtype: int64

In [5]:
train["Exited"].value_counts(normalize=True)

Exited
0.0    0.796267
1.0    0.203733
Name: proportion, dtype: float64

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.countplot(data=train, x="Exited")
plt.title("Баланс классов Exited")
plt.show()

## Шаг 4. Числовые признаки

In [ ]:
continuous_cols = ["CreditScore", "Age", "Balance", "EstimatedSalary"]

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, col in zip(axes.flat, continuous_cols):
    sns.histplot(data=train, x=col, ax=ax)
    ax.set_title(col)
fig.tight_layout()
plt.show()

In [ ]:
discrete_cols = ["Tenure", "NumOfProducts"]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, col in zip(axes, discrete_cols):
    sns.countplot(data=train, x=col, ax=ax)
    ax.set_title(col)
fig.tight_layout()
plt.show()

## Шаг 5. Категориальные признаки и churn rate по группам

In [ ]:
categorical_cols = ["Geography", "Gender", "HasCrCard", "IsActiveMember"]

for col in categorical_cols:
    print(f"=== {col} ===")
    print(train[col].value_counts())
    print(train.groupby(col)["Exited"].mean())
    print()

## Шаг 6. Корреляции

In [ ]:
numeric_cols = [
    "CreditScore", "Age", "Tenure", "Balance",
    "NumOfProducts", "HasCrCard", "IsActiveMember", "EstimatedSalary",
]
corr = train[numeric_cols + ["Exited"]].corr()

print(corr["Exited"].sort_values(ascending=False))

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.show()

## Шаг 7. train vs test — совпадают ли распределения

In [ ]:
train_test = pd.concat([
    train.assign(dataset="train"),
    test.assign(dataset="test"),
])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, col in zip(axes, ["CreditScore", "Age"]):
    sns.histplot(data=train_test, x=col, hue="dataset", stat="density", common_norm=False, ax=ax)
    ax.set_title(col)
fig.tight_layout()
plt.show()

In [ ]:
print(train["EstimatedSalary"].describe())
print()
print(test["EstimatedSalary"].describe())
print()
print(train["EstimatedSalary"].nlargest(5))

## Шаг 8. Выводы

- **Дисбаланс классов**: `Exited=1` — только 20.4% (79.6% / 20.4%). ROC AUC —
  правильный выбор метрики; в туториале 5 обязательно использовать
  `StratifiedKFold`, а не обычный `KFold`.
- **`Age` — самый сильный числовой предиктор**: корреляция с `Exited` ≈ 0.46,
  заметно выше остальных признаков. Логично добавить в feature engineering
  бинирование возраста или нелинейные трансформации.
- **`Geography` — сильный категориальный сигнал**: churn rate в Германии 40.8%,
  почти втрое выше, чем во Франции (15.3%) и Испании (16.6%), при этом клиентов
  из Германии в датасете меньше всего (2812 из 15000). Явный кандидат для
  one-hot/target encoding.
- **`Gender`**: у женщин churn rate 28.3% против 14.3% у мужчин — тоже заметная
  разница, стоит включать признак.
- **`IsActiveMember` подтвердил гипотезу**: неактивные клиенты уходят чаще (28.3%
  против 12.2% у активных) — второй по силе категориальный сигнал после `Geography`.
- **`HasCrCard` почти не влияет**: 22.3% против 19.8% — разница в пределах шума,
  вряд ли окажется полезным признаком.
- **`NumOfProducts` отрицательно коррелирует с оттоком** (-0.29) — в реальных
  данных о банковском оттоке эта зависимость обычно нелинейная (клиенты с 3-4
  продуктами уходят чаще, чем с 2), стоит проверить это отдельно в туториале 4,
  а не полагаться на линейную корреляцию.
- **Найдена аномалия в `EstimatedSalary`**: в train есть 3 экстремальных выброса
  (15.75M, 6.09M, 1.48M), из-за которых std взлетает до 144556 против 45392 в test
  (где максимум ~200K, как и ожидалось для зарплаты). Похоже на артефact
  синтетической генерации данных. **Гипотеза для туториала 4**: перед обучением
  либо клипировать `EstimatedSalary` (например, по 99-му перцентилю), либо
  использовать модели, устойчивые к выбросам (деревья/бустинг и так устойчивы,
  но для baseline с логрегрессией в туториале 3 это может сильно испортить
  масштабирование признака).
- **train и test визуально совпадают** по `CreditScore` и `Age` — локальной
  валидации можно доверять как прокси для реального score.